# Stage B: one net shared by Brent, WTI and six companion series (5, 22 and 66 days)

Runs `experiments/vol_dl_pooled.py` from the `oilfc-residual` Kaggle dataset, as pre-registered in
`docs/adr/0004-deep-learning-addendum.md` (Stage B and the amendment of 2026-11-27). Six cells (two assets, three
horizons); each trains a pooled MLP and a pooled transformer on the anchor residuals of eight series (Brent, WTI, ULSD,
gasoline, jet fuel, propane, Henry Hub, EUR per USD) as QLIKE-trained bounded corrections to the headline benchmark, and
writes its results as soon as it finishes. Set **Accelerator = GPU**, run the cells in order, and download
`dl_pooled_results.zip` from the Output panel. No internet is needed.

In [ ]:
import glob
import os
import shutil
import subprocess
import sys
import zipfile

import torch

print(
    "torch",
    torch.__version__,
    "| cuda:",
    torch.cuda.is_available(),
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",
)

In [ ]:
# 1. Locate the dataset and copy the code to /kaggle/working
WORK = "/kaggle/working"
hits = glob.glob("/kaggle/input/**/experiments/vol_dl_pooled.py", recursive=True)
if hits:
    base = os.path.dirname(os.path.dirname(hits[0]))
else:
    zp = glob.glob("/kaggle/input/**/oilfc-residual.zip", recursive=True)[0]
    base = f"{WORK}/_dataset"
    zipfile.ZipFile(zp).extractall(base)
PROJ = f"{WORK}/proj"
shutil.copytree(base, PROJ, dirs_exist_ok=True)
os.chdir(PROJ)
print(os.listdir(PROJ))

In [ ]:
# 2. Helper and smoke test (a few minutes): 13 monthly splits, 1 seed, 5 epochs. Fix any error here before the full runs.
def run(args):
    p = subprocess.Popen(
        [sys.executable, "-u", "experiments/vol_dl_pooled.py", *args, "--out", f"{WORK}/results"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in p.stdout:
        if "warn" not in line.lower() and "result = " not in line:
            print(line, end="")
    assert p.wait() == 0, "run failed"


run(["--smoke", "--device", "auto"])

In [ ]:
# 3. Brent, 5 and 22 days (5 seeds)
for h in ("22", "5"):
    run(["--asset", "brent", "--h", h, "--seeds", "5", "--device", "auto"])

In [ ]:
# 4. WTI, 5 and 22 days
for h in ("22", "5"):
    run(["--asset", "wti", "--h", h, "--seeds", "5", "--device", "auto"])

In [ ]:
# 5. 66 days, both assets
for asset in ("brent", "wti"):
    run(["--asset", asset, "--h", "66", "--seeds", "5", "--device", "auto"])

In [ ]:
# 6. Package whatever has finished for download (safe to run after any cell)
zp = f"{WORK}/dl_pooled_results.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{WORK}/results/*"):
        z.write(f, os.path.basename(f))
print("ready:", zp, sorted(os.path.basename(f) for f in glob.glob(f"{WORK}/results/*")))